# Notebook 16 – Complete Preprocessing Workflow

## 1. Load Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("hr_employee_attrition_raw.csv")
print("Loaded shape:", df.shape)

Loaded shape: (1230, 16)


## 2. Inspect Dataset

In [3]:
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   EmployeeID          1230 non-null   str    
 1   Age                 1230 non-null   int64  
 2   Gender              1214 non-null   str    
 3   Department          1230 non-null   str    
 4   JobRole             1230 non-null   str    
 5   Education           1194 non-null   str    
 6   MonthlyIncome       1169 non-null   str    
 7   YearsAtCompany      1230 non-null   str    
 8   JobSatisfaction     1179 non-null   float64
 9   PerformanceRating   1230 non-null   int64  
 10  DistanceFromHomeKM  1206 non-null   float64
 11  OverTime            1230 non-null   str    
 12  JoinDate            1230 non-null   str    
 13  Attrition           1230 non-null   str    
 14  EmployeeCountFlag   1230 non-null   int64  
 15  RandomSurveyCode    1230 non-null   int64  
dtypes: float64(2), in

## 3. Identify Data Types

`MonthlyIncome` and `YearsAtCompany` show up as text, not numeric, because
of stray values like "USD" and "yrs" (Notebook 5, Notebook 9). Fixing this
now, since it's a type correction, not a statistical fit.

In [4]:
print(df.dtypes)

df["MonthlyIncome"] = pd.to_numeric(
    df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False), errors="coerce"
)
df["YearsAtCompany"] = pd.to_numeric(
    df["YearsAtCompany"].astype(str).str.replace(" yrs", "", regex=False), errors="coerce"
)
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce")

EmployeeID                str
Age                     int64
Gender                    str
Department                str
JobRole                   str
Education                 str
MonthlyIncome             str
YearsAtCompany            str
JobSatisfaction       float64
PerformanceRating       int64
DistanceFromHomeKM    float64
OverTime                  str
JoinDate                  str
Attrition                 str
EmployeeCountFlag       int64
RandomSurveyCode        int64
dtype: object


## 4. Identify Missing Values

In [5]:
df.isna().sum().sort_values(ascending=False)

MonthlyIncome         61
JobSatisfaction       51
Education             36
DistanceFromHomeKM    24
Gender                16
Age                    0
Department             0
EmployeeID             0
YearsAtCompany         0
JobRole                0
PerformanceRating      0
OverTime               0
JoinDate               0
Attrition              0
EmployeeCountFlag      0
RandomSurveyCode       0
dtype: int64

## 5. Handle Missing Values (Non-Statistical Part Only)

Median/mode imputation is a *learned* value, so that part is deferred to
the pipeline (Step 16), fit only on training data. Here, we only fix
categorical labels that need cleaning regardless of the split, standardizing
`Gender` and `Department` so missing-value counts and category validation
downstream are accurate.

In [6]:
df["Gender"] = df["Gender"].str.strip().str.lower().map(
    {"m": "Male", "male": "Male", "f": "Female", "female": "Female"}
)
df["Department"] = df["Department"].str.strip().str.title()
df["OverTime"] = df["OverTime"].str.strip().str.lower().map(
    {"yes": "Yes", "y": "Yes", "no": "No", "n": "No"}
)

## 6. Detect Duplicates

In [7]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate EmployeeIDs:", df.duplicated(subset=["EmployeeID"]).sum())

Exact duplicate rows: 20
Duplicate EmployeeIDs: 20


## 7. Handle Duplicates

Removing exact duplicates is safe here since we confirmed in Notebook 4
these are true duplicates, not formatting mismatches, after the cleanup in
Step 5.

In [8]:
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (1210, 16)


## 8. Validate Data

Applying the business rules from Notebook 5: age and rating bounds, and
category membership.

In [9]:
print("Invalid ages (outside 15-80):", ((df["Age"] < 15) | (df["Age"] > 80)).sum())
print("Invalid PerformanceRating (outside 1-4):", (~df["PerformanceRating"].between(1, 4)).sum())
print("Rows breaking EmployeeCountFlag constraint:", (df["EmployeeCountFlag"] != 1).sum())
print("Unexpected Gender values:", df[~df["Gender"].isin(["Male", "Female"]) & df["Gender"].notna()].shape[0])

Invalid ages (outside 15-80): 8
Invalid PerformanceRating (outside 1-4): 0
Rows breaking EmployeeCountFlag constraint: 0
Unexpected Gender values: 0


## 9. Detect Outliers

Using the IQR method from Notebook 6 on key numeric columns.

In [10]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Age", "MonthlyIncome", "DistanceFromHomeKM", "YearsAtCompany"]:
    low, high = iqr_bounds(df[col].dropna())
    count = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: bounds=({low:.1f}, {high:.1f}) | outliers={count}")

Age: bounds=(10.0, 58.0) | outliers=16
MonthlyIncome: bounds=(-333.1, 11280.4) | outliers=12
DistanceFromHomeKM: bounds=(-10.6, 23.8) | outliers=72
YearsAtCompany: bounds=(-5.0, 11.0) | outliers=56


## 10. Treat Outliers

Following Notebook 6's per-column judgment, not a blanket rule:

* `Age`: values outside 15-80 are errors, not real, set to missing (to be
  imputed in the pipeline).
* `DistanceFromHomeKM`: capped at 150 km, most flagged rows are real
  long-distance commuters, only the extreme values are broken.
* `MonthlyIncome`: retained as-is, likely real high earners, log
  transformation is applied later for modeling only, not deletion.
* `YearsAtCompany`: retained as-is, high tenure is valid and meaningful.

In [11]:
df.loc[(df["Age"] < 15) | (df["Age"] > 80), "Age"] = np.nan
df["DistanceFromHomeKM"] = df["DistanceFromHomeKM"].clip(upper=150)

## 11. Encode Categorical Variables

Following Notebook 7's guidance: `OneHotEncoder` for nominal, low/medium
cardinality columns (`Gender`, `Department`, `JobRole`, `Education`,
`OverTime`), with `handle_unknown="ignore"` so any category not seen
during training doesn't crash the pipeline later. Fit only on `X_train`.

In [22]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

X_train_cat_encoded = categorical_pipeline.fit_transform(X_train[categorical_features])
X_test_cat_encoded = categorical_pipeline.transform(X_test[categorical_features])

print("Categorical features before encoding:", len(categorical_features))
print("Columns after one-hot encoding:", X_train_cat_encoded.shape[1])

Categorical features before encoding: 5
Columns after one-hot encoding: 29


## 12. Scale Numerical Variables

Following Notebook 8's guidance: `StandardScaler` for the numeric columns,
after imputing missing values with the median (resistant to the outliers
we treated in Notebook 6). Fit only on `X_train`.

In [23]:
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_num_scaled = numeric_pipeline.fit_transform(X_train[numeric_features])
X_test_num_scaled = numeric_pipeline.transform(X_test[numeric_features])

print("Numeric features scaled:", X_train_num_scaled.shape[1])
print("Mean of first scaled column (should be ~0):", X_train_num_scaled[:, 0].mean().round(3))

Numeric features scaled: 6
Mean of first scaled column (should be ~0): -0.0


### Combine Encoded + Scaled Features

Both pieces need to be stacked back into a single feature matrix before
moving on to imbalance handling and feature selection.

In [24]:
import numpy as np

X_train_processed = np.hstack([X_train_num_scaled, X_train_cat_encoded.toarray()])
X_test_processed = np.hstack([X_test_num_scaled, X_test_cat_encoded.toarray()])

print("Combined training shape:", X_train_processed.shape)
print("Combined test shape:", X_test_processed.shape)

Combined training shape: (968, 35)
Combined test shape: (242, 35)


## 13. Handle Class Imbalance

`Attrition` is 81%/19% (Notebook 11). Applying SMOTE only on `X_train`,
never on `X_test`, since resampling test data would distort the honest
evaluation later.

In [25]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_processed, y_train)

print("After SMOTE:", pd.Series(y_train_balanced).value_counts().to_dict())

Before SMOTE: {0: 788, 1: 180}
After SMOTE: {0: 788, 1: 788}


## 14. Perform Feature Selection

Following Notebook 10's guidance: drop zero-variance features first
(quick, cheap filter), since a feature that never changes can't help any
model, no matter how it was encoded or scaled.

In [26]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.0)
X_train_selected = selector.fit_transform(X_train_balanced)
X_test_selected = selector.transform(X_test_processed)

print("Features before selection:", X_train_balanced.shape[1])
print("Features after selection:", X_train_selected.shape[1])

Features before selection: 35
Features after selection: 35


## 15. Split Data

Splitting now, before any statistical fitting happens, using stratification
since `Attrition` is imbalanced

In [27]:
from sklearn.model_selection import train_test_split

df["Attrition_binary"] = df["Attrition"].map({"Yes": 1, "No": 0})

numeric_features = ["Age", "MonthlyIncome", "YearsAtCompany", "DistanceFromHomeKM",
                      "JobSatisfaction", "PerformanceRating"]
categorical_features = ["Gender", "Department", "JobRole", "Education", "OverTime"]

X = df[numeric_features + categorical_features]
y = df["Attrition_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (968, 11) | Test: (242, 11)


## 16. Build Preprocessing Pipeline

Combining imputation, scaling, and encoding into one `ColumnTransformer`
(Notebook 14), fit only on `X_train`.

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit ONLY on training data, transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (968, 35)
Processed test shape: (242, 35)


## 17. Validate Final Dataset

Confirming the dataset is genuinely ML-ready: no missing values, no
duplicate rows, numeric types throughout, and balanced classes for
training.

In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from imblearn.over_sampling import SMOTE

# --- Step 15: Split Data ---
df["Attrition_binary"] = df["Attrition"].map({"Yes": 1, "No": 0})

numeric_features = ["Age", "MonthlyIncome", "YearsAtCompany", "DistanceFromHomeKM",
                      "JobSatisfaction", "PerformanceRating"]
categorical_features = ["Gender", "Department", "JobRole", "Education", "OverTime"]

X = df[numeric_features + categorical_features]
y = df["Attrition_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Step 16: Build Preprocessing Pipeline + Feature Selection ---
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

selector = VarianceThreshold(threshold=0.0)
X_train_selected = selector.fit_transform(X_train_processed)
X_test_selected = selector.transform(X_test_processed)

# --- Step 13: Handle Class Imbalance ---
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)

# --- Step 17: Validate Final Dataset ---
X_train_final = pd.DataFrame(X_train_balanced)
X_test_final = pd.DataFrame(X_test_selected)

print("Final training set shape:", X_train_final.shape)
print("Final test set shape:", X_test_final.shape)
print("Any missing values in training set:", X_train_final.isna().sum().sum())
print("Any missing values in test set:", X_test_final.isna().sum().sum())
print("Duplicate rows in training set:", X_train_final.duplicated().sum())
print("Final training class balance:", pd.Series(y_train_balanced).value_counts(normalize=True).round(3).to_dict())
print("Test set class balance (untouched, for honest evaluation):", pd.Series(y_test).value_counts(normalize=True).round(3).to_dict())

Final training set shape: (1576, 35)
Final test set shape: (242, 35)
Any missing values in training set: 0
Any missing values in test set: 0
Duplicate rows in training set: 6
Final training class balance: {0: 0.5, 1: 0.5}
Test set class balance (untouched, for honest evaluation): {0: 0.814, 1: 0.186}
